# Cross-venue path confirmation — strict 42-fold audit

## TL;DR

The pre-registered cross-venue meta-label is **rejected before strategy integration**. It scored 34 feed-forward folds / 496 terminal conditions. Market-minus-meta Brier improvement was `-0.001597` and log-loss improvement was `-0.003363`. No live or shadow promotion follows from this mechanism audit.

## Method

The only candidate family is a ridge-regularized logistic meta-label with executable-market log-odds as its offset. Inputs are the chosen token's causal logit change and the direction-aligned BTC log return at fixed 5, 30, and 60 second horizons. Ridge is selected on the final two strictly prior folds and refit on all prior folds. Each terminal condition has total weight one. The full cohort was not inspected before this family and its gates were frozen.

In [1]:
from __future__ import annotations

import collections
import datetime as dt
import json
import math
import re
from pathlib import Path

import numpy as np

OPPORTUNITY_DIR = Path(
    "/private/tmp/polymomentum_strategy_a_plus_20260715/"
    "path_confirmation_opportunities_strict42_latency202"
)
BASELINE_DIR = Path(
    "/private/tmp/polymomentum_strategy_a_plus_20260714/"
    "volfloor_strict42_latency202_exposure8"
)
EVIDENCE_PATH = Path(
    "/Users/ttoomm/Documents/PolyMomentum/deploy/promotions/evidence/strategy_registry/"
    "20260715_cross_venue_path_confirmation_strict42.json"
)
EXPECTED_LATENCY_MS = 202
BASELINE_PARAMS_HASH = "a5d67641653ae85a853aab531060a240eade257e32fd5bf0e46392c7934302d5"
MIN_PRIOR_FOLDS = 8
VALIDATION_FOLDS = 2
RIDGES = (0.01, 0.1, 1.0, 10.0)
FEATURE_NAMES = (
    "token_logit_change_5s",
    "directional_btc_return_bps_5s",
    "token_logit_change_30s",
    "directional_btc_return_bps_30s",
    "token_logit_change_60s",
    "directional_btc_return_bps_60s",
)
EPSILON = 1e-6


## Data quality and causal feature coverage

In [2]:
def load_opportunities():
    files = sorted(OPPORTUNITY_DIR.glob("fold_*_opportunities.json"))
    assert len(files) == 42, f"expected 42 opportunity files, found {len(files)}"
    rows = []
    source_folds = []
    for path in files:
        match = re.match(r"fold_(\d+)_", path.name)
        assert match, path
        fold = int(match.group(1))
        payload = json.loads(path.read_text())
        report = json.loads(
            path.with_name(path.name.replace("_opportunities.json", "_report.json")).read_text()
        )
        variant = report["variants"][0]
        assert payload["continuous"] is True
        assert payload["latency_ms"] == EXPECTED_LATENCY_MS
        assert payload["variant_count"] == 1
        assert payload["data_manifest"]["complete"] is True
        assert payload["row_count"] == len(payload["rows"])
        assert variant["trades"] == 0 and variant["execution_attempts"] == 0
        source_folds.append(
            {
                "fold": fold,
                "rows": payload["row_count"],
                "conditions": payload["condition_count"],
                "manifest_hash": payload["data_manifest"]["manifest_hash"],
            }
        )
        for wrapped in payload["rows"]:
            opportunity = dict(wrapped["opportunity"])
            decision = opportunity.pop("decision")
            opportunity.update(
                {
                    "fold": fold,
                    "fair_value": decision["fair_value"],
                    "market_price": decision["market_price"],
                    "minutes_remaining": decision["minutes_remaining"],
                    "z_score": decision["z_score"],
                    "direction": decision["direction"],
                    "zone": decision["zone"],
                    "row_index": len(rows),
                }
            )
            rows.append(opportunity)
    return rows, source_folds


all_rows, source_folds = load_opportunities()
folds = sorted({row["fold"] for row in all_rows})
raw_condition_counts = collections.Counter(row["condition_id"] for row in all_rows)
condition_directions = collections.defaultdict(set)
condition_folds = collections.defaultdict(set)
condition_seconds = []
for row in all_rows:
    condition_directions[row["condition_id"]].add(row["actual_direction"])
    condition_folds[row["condition_id"]].add(row["fold"])
    condition_seconds.append((row["condition_id"], row["sampling_second"]))

feature_coverage = {
    feature: sum(row[feature] is not None and math.isfinite(row[feature]) for row in all_rows)
    for feature in FEATURE_NAMES
}
rows = [
    row
    for row in all_rows
    if all(row[feature] is not None and math.isfinite(row[feature]) for feature in FEATURE_NAMES)
]
for row_index, row in enumerate(rows):
    row["row_index"] = row_index
condition_counts = collections.Counter(row["condition_id"] for row in rows)
quality = {
    "files": len(source_folds),
    "folds": len(folds),
    "rows": len(all_rows),
    "conditions": len(raw_condition_counts),
    "complete_case_rows": len(rows),
    "complete_case_conditions": len(condition_counts),
    "complete_case_fraction": len(rows) / len(all_rows),
    "duplicate_condition_seconds": len(condition_seconds) - len(set(condition_seconds)),
    "condition_direction_conflicts": sum(len(v) != 1 for v in condition_directions.values()),
    "conditions_crossing_folds": sum(len(v) != 1 for v in condition_folds.values()),
    "outcome_mapping_mismatches": sum(
        row["won"] != (row["direction"] == row["actual_direction"]) for row in all_rows
    ),
    "sampling_floor_mismatches": sum(
        math.floor(row["decision_timestamp_s"]) != row["sampling_second"] for row in all_rows
    ),
    "feature_coverage": feature_coverage,
    "feature_coverage_fraction": {
        feature: count / len(all_rows) for feature, count in feature_coverage.items()
    },
    "resolution_sources": dict(
        collections.Counter(row["resolution_source"] for row in all_rows)
    ),
}
assert quality["folds"] == 42
assert quality["duplicate_condition_seconds"] == 0
assert quality["condition_direction_conflicts"] == 0
assert quality["conditions_crossing_folds"] == 0
assert quality["outcome_mapping_mismatches"] == 0
assert quality["sampling_floor_mismatches"] == 0
assert quality["complete_case_fraction"] >= 0.99


## Strictly prior-fold meta-label

Feature standardization, coefficient fitting, and ridge selection use prior folds only. The market is an offset rather than a replaceable feature, so the model must add information beyond the executable market to pass.

In [3]:
def clip_probability(probability):
    return np.clip(np.asarray(probability, dtype=float), EPSILON, 1.0 - EPSILON)


def logit(probability):
    probability = clip_probability(probability)
    return np.log(probability / (1.0 - probability))


def sigmoid(value):
    value = np.asarray(value, dtype=float)
    out = np.empty_like(value)
    positive = value >= 0.0
    out[positive] = 1.0 / (1.0 + np.exp(-value[positive]))
    exponential = np.exp(value[~positive])
    out[~positive] = exponential / (1.0 + exponential)
    return out


def metric_pair(outcomes, probabilities, weights):
    probabilities = clip_probability(probabilities)
    normalized = np.asarray(weights, dtype=float)
    normalized = normalized / normalized.sum()
    return {
        "brier": float(np.sum(normalized * (probabilities - outcomes) ** 2)),
        "log_loss": float(
            -np.sum(
                normalized
                * (
                    outcomes * np.log(probabilities)
                    + (1.0 - outcomes) * np.log(1.0 - probabilities)
                )
            )
        ),
    }


def fit_standardizer(features, weights):
    normalized = weights / weights.sum()
    mean = np.sum(normalized[:, None] * features, axis=0)
    variance = np.sum(normalized[:, None] * (features - mean) ** 2, axis=0)
    scale = np.sqrt(np.maximum(variance, 1e-8))
    return mean, scale


def standardized_design(features, mean, scale):
    standardized = np.clip((features - mean) / scale, -5.0, 5.0)
    return np.column_stack([np.ones(len(standardized)), standardized])


def penalized_objective(outcomes, offset, design, weights, coefficients, ridge):
    probabilities = clip_probability(sigmoid(offset + design @ coefficients))
    normalized = weights / weights.sum()
    loss = -np.sum(
        normalized
        * (
            outcomes * np.log(probabilities)
            + (1.0 - outcomes) * np.log(1.0 - probabilities)
        )
    )
    return float(loss + 0.5 * ridge * np.dot(coefficients[1:], coefficients[1:]))


def fit_meta_label(market, features, outcomes, weights, ridge):
    mean, scale = fit_standardizer(features, weights)
    design = standardized_design(features, mean, scale)
    offset = logit(market)
    coefficients = np.zeros(design.shape[1])
    penalty = np.diag([0.0] + [ridge] * features.shape[1])
    lower = np.array([-1.5] + [-2.0] * features.shape[1])
    upper = np.array([1.5] + [2.0] * features.shape[1])
    normalized = weights / weights.sum()
    for _ in range(100):
        eta = offset + design @ coefficients
        probabilities = sigmoid(eta)
        gradient = design.T @ (normalized * (probabilities - outcomes)) + penalty @ coefficients
        curvature = normalized * probabilities * (1.0 - probabilities)
        hessian = design.T @ (curvature[:, None] * design) + penalty + 1e-8 * np.eye(design.shape[1])
        step = np.linalg.solve(hessian, gradient)
        step = np.clip(np.nan_to_num(step, nan=0.0, posinf=10.0, neginf=-10.0), -10.0, 10.0)
        old = penalized_objective(outcomes, offset, design, weights, coefficients, ridge)
        scale_factor = 1.0
        accepted = False
        while scale_factor >= 1e-6:
            candidate = np.clip(coefficients - scale_factor * step, lower, upper)
            if (
                penalized_objective(outcomes, offset, design, weights, candidate, ridge)
                <= old + 1e-12
            ):
                accepted = True
                break
            scale_factor *= 0.5
        if not accepted or np.max(np.abs(candidate - coefficients)) < 1e-9:
            break
        coefficients = candidate
    return {"mean": mean, "scale": scale, "coefficients": coefficients, "ridge": ridge}


def predict_meta_label(model, market, features):
    design = standardized_design(features, model["mean"], model["scale"])
    return sigmoid(logit(market) + design @ model["coefficients"])


n = len(rows)
fold_array = np.array([row["fold"] for row in rows], dtype=int)
condition_array = np.array([row["condition_id"] for row in rows], dtype=object)
outcomes = np.array([row["won"] for row in rows], dtype=float)
market = np.array([row["market_price"] for row in rows], dtype=float)
features = np.array([[row[feature] for feature in FEATURE_NAMES] for row in rows], dtype=float)
weights = np.array([1.0 / condition_counts[row["condition_id"]] for row in rows], dtype=float)
predictions = np.full(n, np.nan)
fold_models = []

for position in range(MIN_PRIOR_FOLDS, len(folds)):
    scored_fold = folds[position]
    prior_folds = folds[:position]
    validation_folds = prior_folds[-VALIDATION_FOLDS:]
    fit_folds = prior_folds[:-VALIDATION_FOLDS]
    fit_mask = np.isin(fold_array, fit_folds)
    validation_mask = np.isin(fold_array, validation_folds)
    ridge_choices = []
    for ridge in RIDGES:
        fitted = fit_meta_label(
            market[fit_mask], features[fit_mask], outcomes[fit_mask], weights[fit_mask], ridge
        )
        validation_prediction = predict_meta_label(
            fitted, market[validation_mask], features[validation_mask]
        )
        score = metric_pair(outcomes[validation_mask], validation_prediction, weights[validation_mask])
        ridge_choices.append((score["log_loss"], score["brier"], ridge))
    _, _, selected_ridge = min(ridge_choices)
    prior_mask = np.isin(fold_array, prior_folds)
    fitted = fit_meta_label(
        market[prior_mask], features[prior_mask], outcomes[prior_mask], weights[prior_mask], selected_ridge
    )
    scored_mask = fold_array == scored_fold
    predictions[scored_mask] = predict_meta_label(fitted, market[scored_mask], features[scored_mask])
    fold_models.append(
        {
            "fold": scored_fold,
            "prior_folds": len(prior_folds),
            "validation_folds": validation_folds,
            "ridge": selected_ridge,
            "feature_means": dict(zip(FEATURE_NAMES, fitted["mean"].tolist())),
            "feature_scales": dict(zip(FEATURE_NAMES, fitted["scale"].tolist())),
            "intercept": float(fitted["coefficients"][0]),
            "standardized_coefficients": dict(
                zip(FEATURE_NAMES, fitted["coefficients"][1:].tolist())
            ),
        }
    )

scored = np.isfinite(predictions)
scored_folds = [item["fold"] for item in fold_models]


## Proper-score validation and feature separation

In [4]:
def comparison(mask):
    market_score = metric_pair(outcomes[mask], market[mask], weights[mask])
    meta_score = metric_pair(outcomes[mask], predictions[mask], weights[mask])
    return {
        "conditions": len(set(condition_array[mask])),
        "rows": int(mask.sum()),
        "market": market_score,
        "meta_label": meta_score,
        "market_minus_meta_brier": market_score["brier"] - meta_score["brier"],
        "market_minus_meta_log_loss": market_score["log_loss"] - meta_score["log_loss"],
    }


overall = comparison(scored)
half = len(scored_folds) // 2
first_half_mask = scored & np.isin(fold_array, scored_folds[:half])
second_half_mask = scored & np.isin(fold_array, scored_folds[half:])
chronological_halves = {
    "first": comparison(first_half_mask),
    "second": comparison(second_half_mask),
}
fold_scores = []
for fold in scored_folds:
    item = comparison(scored & (fold_array == fold))
    item["fold"] = fold
    fold_scores.append(item)

condition_differences = collections.defaultdict(lambda: {"brier": [], "log_loss": []})
for index in np.flatnonzero(scored):
    market_probability = float(clip_probability(market[index]))
    meta_probability = float(clip_probability(predictions[index]))
    outcome = outcomes[index]
    condition_differences[condition_array[index]]["brier"].append(
        (market_probability - outcome) ** 2 - (meta_probability - outcome) ** 2
    )
    market_loss = -(
        outcome * math.log(market_probability) + (1.0 - outcome) * math.log(1.0 - market_probability)
    )
    meta_loss = -(
        outcome * math.log(meta_probability) + (1.0 - outcome) * math.log(1.0 - meta_probability)
    )
    condition_differences[condition_array[index]]["log_loss"].append(market_loss - meta_loss)

bootstrap_conditions = sorted(condition_differences)
condition_brier = np.array(
    [np.mean(condition_differences[condition]["brier"]) for condition in bootstrap_conditions]
)
condition_log_loss = np.array(
    [np.mean(condition_differences[condition]["log_loss"]) for condition in bootstrap_conditions]
)
rng = np.random.default_rng(20260715)
bootstrap_brier = np.empty(2000)
bootstrap_log_loss = np.empty(2000)
for index in range(2000):
    draw = rng.integers(0, len(bootstrap_conditions), len(bootstrap_conditions))
    bootstrap_brier[index] = np.mean(condition_brier[draw])
    bootstrap_log_loss[index] = np.mean(condition_log_loss[draw])
bootstrap = {
    "market_minus_meta_brier_95pct": np.quantile(bootstrap_brier, [0.025, 0.975]).tolist(),
    "market_minus_meta_log_loss_95pct": np.quantile(
        bootstrap_log_loss, [0.025, 0.975]
    ).tolist(),
}

feature_separation = {}
for feature_index, feature_name in enumerate(FEATURE_NAMES):
    values = features[scored, feature_index]
    scored_outcomes = outcomes[scored]
    scored_weights = weights[scored]
    win_mask = scored_outcomes == 1.0
    loss_mask = ~win_mask
    win_mean = float(np.average(values[win_mask], weights=scored_weights[win_mask]))
    loss_mean = float(np.average(values[loss_mask], weights=scored_weights[loss_mask]))
    pooled_std = float(np.sqrt(np.average((values - np.average(values, weights=scored_weights)) ** 2, weights=scored_weights)))
    feature_separation[feature_name] = {
        "condition_weighted_win_mean": win_mean,
        "condition_weighted_loss_mean": loss_mean,
        "win_minus_loss": win_mean - loss_mean,
        "standardized_win_minus_loss": (win_mean - loss_mean) / max(pooled_std, 1e-8),
    }


## Baseline loss-cluster diagnostic

The exact floor-0.30 trade cohort is matched to the nearest same-condition, same-direction opportunity within one second. This is descriptive only; it cannot authorize a counterfactual guard.

In [5]:
def load_baseline_trades():
    trades = []
    for path in sorted(BASELINE_DIR.glob("fold_*_features.json")):
        match = re.match(r"fold_(\d+)_", path.name)
        assert match, path
        fold = int(match.group(1))
        payload = json.loads(path.read_text())
        for row in payload["rows"]:
            if row["params_hash"] == BASELINE_PARAMS_HASH:
                item = dict(row)
                item["fold"] = fold
                trades.append(item)
    return trades


baseline_trades = load_baseline_trades()
opportunities_by_condition = collections.defaultdict(list)
for row in rows:
    opportunities_by_condition[row["condition_id"]].append(row)

matched_trade_rows = []
unmatched_trades = []
for trade in baseline_trades:
    candidates = [
        row
        for row in opportunities_by_condition[trade["condition_id"]]
        if row["direction"] == trade["decision"]["direction"]
    ]
    if not candidates:
        unmatched_trades.append({"condition_id": trade["condition_id"], "fold": trade["fold"]})
        continue
    match = min(
        candidates,
        key=lambda row: abs(row["decision_timestamp_s"] - trade["decision_timestamp_s"]),
    )
    gap = abs(match["decision_timestamp_s"] - trade["decision_timestamp_s"])
    if gap > 1.0:
        unmatched_trades.append(
            {"condition_id": trade["condition_id"], "fold": trade["fold"], "nearest_gap_s": gap}
        )
        continue
    item = {
        "fold": trade["fold"],
        "condition_id": trade["condition_id"],
        "won": trade["won"],
        "pnl_after_fee": trade["pnl_after_fee"],
        "decision_timestamp_s": trade["decision_timestamp_s"],
        "opportunity_timestamp_s": match["decision_timestamp_s"],
        "timestamp_gap_s": gap,
        "row_index": match["row_index"],
        "market_price": match["market_price"],
        "feature_values": {feature: match[feature] for feature in FEATURE_NAMES},
    }
    if np.isfinite(predictions[match["row_index"]]):
        item["meta_probability"] = float(predictions[match["row_index"]])
        item["meta_minus_market"] = item["meta_probability"] - item["market_price"]
    matched_trade_rows.append(item)

scored_trades = [trade for trade in matched_trade_rows if "meta_minus_market" in trade]
win_corrections = [trade["meta_minus_market"] for trade in scored_trades if trade["won"]]
loss_corrections = [trade["meta_minus_market"] for trade in scored_trades if not trade["won"]]
pairwise_auc = (
    float(
        np.mean(
            [
                float(win > loss) + 0.5 * float(win == loss)
                for win in win_corrections
                for loss in loss_corrections
            ]
        )
    )
    if win_corrections and loss_corrections
    else None
)
baseline_trade_diagnostic = {
    "baseline_trades": len(baseline_trades),
    "matched_trades": len(matched_trade_rows),
    "unmatched_trades": unmatched_trades,
    "maximum_timestamp_gap_s": max(
        (trade["timestamp_gap_s"] for trade in matched_trade_rows), default=None
    ),
    "scored_trades": len(scored_trades),
    "scored_wins": sum(trade["won"] for trade in scored_trades),
    "scored_losses": sum(not trade["won"] for trade in scored_trades),
    "mean_meta_minus_market_wins": float(np.mean(win_corrections)) if win_corrections else None,
    "mean_meta_minus_market_losses": float(np.mean(loss_corrections)) if loss_corrections else None,
    "win_vs_loss_pairwise_auc": pairwise_auc,
}

trade_feature_separation = {}
for feature_name in FEATURE_NAMES:
    win_values = [
        trade["feature_values"][feature_name] for trade in matched_trade_rows if trade["won"]
    ]
    loss_values = [
        trade["feature_values"][feature_name] for trade in matched_trade_rows if not trade["won"]
    ]
    trade_feature_separation[feature_name] = {
        "win_mean": float(np.mean(win_values)),
        "loss_mean": float(np.mean(loss_values)),
        "win_median": float(np.median(win_values)),
        "loss_median": float(np.median(loss_values)),
        "win_minus_loss_mean": float(np.mean(win_values) - np.mean(loss_values)),
    }


## Evidence export and verdict

In [6]:
support_ok = len(scored_folds) >= 30 and overall["conditions"] >= 300
overall_ok = overall["market_minus_meta_brier"] > 0.0 and overall["market_minus_meta_log_loss"] > 0.0
halves_ok = all(
    half_result["market_minus_meta_brier"] > 0.0
    and half_result["market_minus_meta_log_loss"] > 0.0
    for half_result in chronological_halves.values()
)
bootstrap_ok = (
    bootstrap["market_minus_meta_brier_95pct"][0] > 0.0
    and bootstrap["market_minus_meta_log_loss_95pct"][0] > 0.0
)
eligible_for_exact_replay = support_ok and overall_ok and halves_ok and bootstrap_ok
failure_reasons = []
if not support_ok:
    failure_reasons.append("insufficient_scored_support")
if not overall_ok:
    failure_reasons.append("proper_score_improvement_failed_overall")
if not halves_ok:
    failure_reasons.append("proper_score_improvement_failed_chronological_half")
if not bootstrap_ok:
    failure_reasons.append("condition_bootstrap_lower_bound_not_positive")

summary = {
    "schema_version": 1,
    "generated_at": dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z"),
    "status": "eligible_for_exact_replay" if eligible_for_exact_replay else "rejected_before_strategy_integration",
    "promotion_eligible": False,
    "promotion_posture": "research_only_live_off",
    "methodology": [
        "Use one pre-registered ridge-regularized logistic meta-label with executable-market log-odds as an offset.",
        "Use only six causal features: chosen-token logit changes and direction-aligned BTC log returns at 5, 30, and 60 seconds.",
        "For each scored fold, select ridge on the final two strictly prior folds, refit on all prior folds, and score the next fold once.",
        "Give every terminal condition total weight one so repeated one-second opportunities do not create pseudo-replication.",
        "Require positive Brier and log-loss improvement overall, in both chronological halves, and at the 95 percent condition-bootstrap lower bound before exact replay.",
        "Treat baseline-trade feature matching as descriptive loss-cluster diagnosis only; it cannot authorize strategy integration.",
    ],
    "data_quality": quality,
    "model": {
        "features": list(FEATURE_NAMES),
        "offset": "logit(executable_market_price)",
        "ridge_candidates": list(RIDGES),
        "minimum_prior_folds": MIN_PRIOR_FOLDS,
        "validation_folds": VALIDATION_FOLDS,
        "standardized_feature_clip": 5.0,
        "coefficient_bounds": {"intercept": [-1.5, 1.5], "features": [-2.0, 2.0]},
    },
    "result": {
        "eligible_for_exact_replay": eligible_for_exact_replay,
        "support_ok": support_ok,
        "overall_ok": overall_ok,
        "chronological_halves_ok": halves_ok,
        "bootstrap_ok": bootstrap_ok,
        "failure_reasons": failure_reasons,
        "scored_folds": len(scored_folds),
        "scored_conditions": overall["conditions"],
    },
    "overall": overall,
    "chronological_halves": chronological_halves,
    "condition_bootstrap_95pct": bootstrap,
    "feature_separation": feature_separation,
    "baseline_trade_diagnostic": baseline_trade_diagnostic,
    "baseline_trade_feature_separation": trade_feature_separation,
    "selected_ridge_counts": dict(collections.Counter(item["ridge"] for item in fold_models)),
    "fold_models": fold_models,
    "fold_scores": fold_scores,
    "source_folds": source_folds,
    "verdict": (
        "The cross-venue meta-label passed its pre-registered proper-score screen and may proceed to exact strategy replay."
        if eligible_for_exact_replay
        else "The cross-venue meta-label failed its pre-registered proper-score screen; do not integrate or exact-replay it."
    ),
}

EVIDENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_path = EVIDENCE_PATH.with_name(EVIDENCE_PATH.name + ".tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
temporary_path.replace(EVIDENCE_PATH)

print(json.dumps({"quality": quality, "result": summary["result"], "overall": overall, "chronological_halves": chronological_halves, "bootstrap": bootstrap, "baseline_trade_diagnostic": baseline_trade_diagnostic}, indent=2, sort_keys=True))


{
  "baseline_trade_diagnostic": {
    "baseline_trades": 102,
    "matched_trades": 101,
    "maximum_timestamp_gap_s": 0.8920001983642578,
    "mean_meta_minus_market_losses": 0.04302532453968586,
    "mean_meta_minus_market_wins": 0.024599738189535775,
    "scored_losses": 20,
    "scored_trades": 85,
    "scored_wins": 65,
    "unmatched_trades": [
      {
        "condition_id": "0xd38779f8bc8a4d61c0ceb852094312f8ba2204f7946433d6ba28a0062ebb4807",
        "fold": 26,
        "nearest_gap_s": 24.057999849319458
      }
    ],
    "win_vs_loss_pairwise_auc": 0.5392307692307692
  },
  "bootstrap": {
    "market_minus_meta_brier_95pct": [
      -0.004264185251840156,
      0.0007912380931120234
    ],
    "market_minus_meta_log_loss_95pct": [
      -0.011676858528055663,
      0.004032146163804672
    ]
  },
  "chronological_halves": {
    "first": {
      "conditions": 244,
      "market": {
        "brier": 0.13529209765598108,
        "log_loss": 0.43929429678582926
      },
      